# Klaatch Topic Messages Analysis

## Overview

This notebook analyzes top messages for each topic category in the Klaatch dataset. It:
1. Loads topic features from DLATK categorical features table
2. Merges with message data and metadata
3. Identifies top 10 messages per topic category based on group_norm scores
4. Associates top words with each topic
5. Exports results for interpretation and analysis
6. Runs DLATK predictions using topic features

## Outputs
- **CSV file**: Top 10 messages per topic with associated words
- **DLATK predictions**: CEL score predictions using topic features

---

## 1. Setup and Database Connection

Import libraries and establish connection to MySQL database.

In [ ]:
import pandas as pd
import json
import sqlalchemy
from sqlalchemy import create_engine, text
import os

# Database configuration
db = sqlalchemy.engine.url.URL(
    drivername='mysql',
    host='***********',
    database='Audio_features',
    username='************',
    password='************',
    port=None,
    query={'read_default_file': '~/.my.cnf', 'charset': 'utf8mb4'}
)

# Create engine
engine = sqlalchemy.create_engine(db)
print('✓ Database connection established')

## 2. Configuration

Define table names and data sources:
- **Feature table**: DLATK categorical features (100 topic categories)
- **Message table**: Main data table with messages and CEL scores
- **Topic words**: Pre-computed top 10 words per topic

In [ ]:
# Table names
FEATURE_TABLE = 'feat$cat_ZClaatch_100_cp_w$merged_data$message_id$1gra'
MESSAGE_TABLE = 'merged_data'
TOPIC_WORDS_FILE = '/home/karthik9/karthik_klutch/50TOPICS.csv'

print(f'Feature table: {FEATURE_TABLE}')
print(f'Message table: {MESSAGE_TABLE}')
print(f'Topic words file: {TOPIC_WORDS_FILE}')

## 3. Load Data

Load topic features, message data, and top words from database and CSV files.

In [ ]:
# Load feature table (topic categories)
print('Loading feature table...')
tbl = pd.read_sql_table(FEATURE_TABLE, engine)
print(f'✓ Loaded {len(tbl)} feature records')

# Load message table
print('\nLoading message table...')
msg_tbl = pd.read_sql_table(MESSAGE_TABLE, engine)
msg_tbl = msg_tbl[[
    'message_id', 'message', 'KlaatchID_x', 'Date_x',
    'CEL_Total', 'CELVAL1', 'CELVAL2', 'CELVAL3'
]]
print(f'✓ Loaded {len(msg_tbl)} messages')

# Load top words by topic
print('\nLoading topic words...')
top_10_words_by_topic = pd.read_csv(TOPIC_WORDS_FILE)
print(f'✓ Loaded top words for {len(top_10_words_by_topic)} topics')

print('\nData loading complete!')

## 4. Preprocess Features

Clean and prepare the feature table:
- Remove intercept term
- Convert feature IDs to integers (topic categories)
- Rename for clarity

In [ ]:
# Remove intercept
print(f'Records before filtering: {len(tbl)}')
tbl = tbl[tbl['feat'] != '_intercept']
print(f'Records after removing intercept: {len(tbl)}')

# Convert feat to integer (topic category)
tbl['feat'] = tbl['feat'].astype(int)

# Rename for clarity
tbl = tbl.rename(columns={'feat': 'category'})

print(f'\nTopic categories: {tbl["category"].min()} to {tbl["category"].max()}')
print(f'Unique categories: {tbl["category"].nunique()}')

tbl.head()

## 5. Merge Features with Messages

Combine topic features with message text and metadata.

In [ ]:
# Merge features with messages
print('Merging features with messages...')
tbl = tbl.merge(msg_tbl, left_on='group_id', right_on='message_id', how='left')

print(f'✓ Merged data: {len(tbl)} records')
print(f'Columns: {list(tbl.columns)}')

# Check for missing messages
missing = tbl['message'].isna().sum()
print(f'\nRecords with missing messages: {missing}')

tbl.head()

## 6. Filter Messages by Length

Create subset of messages with more than 5 words for quality filtering.

In [ ]:
# Filter messages with more than 5 words
tbl_5_words = tbl[tbl['message'].str.split().str.len() > 5]

print(f'Original records: {len(tbl)}')
print(f'Records with >5 words: {len(tbl_5_words)}')
print(f'Filtered out: {len(tbl) - len(tbl_5_words)} records ({(len(tbl) - len(tbl_5_words))/len(tbl)*100:.1f}%)')

# Note: Using tbl (not tbl_5_words) for top messages to include all data
print('\nNote: Using all messages (not filtered) for top message extraction')

## 7. Extract Top Messages per Topic

For each topic category, extract the 10 messages with highest group_norm scores.
The group_norm score indicates how strongly a message represents that topic.

In [ ]:
# Get top 10 messages per category
print('Extracting top 10 messages per category...')
top_10_msgs = tbl.groupby('category').apply(
    lambda x: x.nlargest(10, 'group_norm')
).reset_index(drop=True)

print(f'✓ Extracted {len(top_10_msgs)} top messages')
print(f'Categories: {top_10_msgs["category"].nunique()}')
print(f'Messages per category: {len(top_10_msgs) / top_10_msgs["category"].nunique():.1f} avg')

top_10_msgs.head()

## 8. Associate Topic Words with Messages

Merge top messages with their corresponding topic words for interpretation.

In [ ]:
# Merge top messages with topic words
print('Merging messages with topic words...')
top_10_msgs = top_10_msgs.merge(top_10_words_by_topic, on='category', how='left')

# Select relevant columns
result = top_10_msgs[['category', 'message_id', 'message', 'top_10_words']]

print(f'✓ Final result: {len(result)} records')
print('\nSample:')
result.head()

## 9. Export Results

Save the top messages with associated topic words to CSV file for analysis and interpretation.

In [ ]:
# Export to CSV
output_file = 'z100.csv'
result.to_csv(output_file, index=False)

print(f'✓ Results saved to: {output_file}')
print(f'\nFile contains:')
print(f'  - {result["category"].nunique()} topic categories')
print(f'  - {len(result)} messages (top 10 per category)')
print(f'  - Associated topic words for each category')

# Display sample
print('\nSample of results:')
result.head(20)

## 10. DLATK Prediction Analysis

Run DLATK ridge regression predictions using topic features to predict CEL scores.

**Configuration:**
- **Features**: Topic categories (100 LDA topics)
- **Outcomes**: CEL_Total, CELVAL1, CELVAL2, CELVAL3
- **Model**: Ridge regression with high cross-validation
- **Validation**: N-fold test with fold column
- **Threshold**: Minimum 10 messages per group

In [ ]:
!python /home/karthik9/TheDlatk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f  'feat$cat_Klaatch_lda_100_cp_w$merged_data$message_id$1gra'  \
    --outcome_table merged_data  --group_freq_thresh 10 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold